# 19 — Arbres et ping-pong : sous-groupes libres de SL₂(ℤ)

Après le versant arithmétique de la série (jusqu'au carnet 18 sur Dedekind), ce carnet ouvre
un nouveau versant : **« Arbres, amalgames et SL₂ » (1977)**, le livre où Serre fait agir
$SL_2(\mathbb{Z})$ sur un arbre — et où la théorie des groupes y découvre ses sous-groupes libres.

La question centrale est ancienne et précise : $SL_2(\mathbb{Z})$ est-il « gros » ? Contient-il un
sous-groupe libre de rang 2 ? La réponse — le théorème de Sanov (1947) — s'obtient par le **lemme
de ping-pong**, l'une des démonstrations les plus jouables de l'algèbre : deux matrices échangent
deux parties disjointes de la droite projective, donc aucune relation ne peut exister entre elles.

Tout ce carnet est **calculé en arithmétique exacte** (vecteurs primitifs, `fractions.Fraction`),
sans aucune bibliothèque au-delà de la stdlib. Chaque énoncé central — les ordres des générateurs,
la transitivité de l'action sur $P^1(\mathbb{Q})$, la structure d'arbre du pavage de Farey, la
disjonction du ping-pong, la liberté mesurée — est **mesuré dans une sortie**.

## 1. Trois générateurs, une signature

On travaille dans $SL_2(\mathbb{Z})$ : matrices $2\times 2$ à coefficients entiers, déterminant 1.
Trois éléments suffisent à tout engendrer :

$$S = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}, \quad T = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}, \quad R = S\,T = \begin{pmatrix} 0 & -1 \\ 1 & 1 \end{pmatrix}.$$

Sur la droite projective, $S$ agit par $z \mapsto -1/z$ (inversion) et $T$ par $z \mapsto z + 1$
(translation). Mesurons leurs ordres exacts et leurs relations :

In [1]:
from fractions import Fraction
from math import gcd

def matmul(A, B):
    """Produit matriciel exact en entiers."""
    return [[sum(A[i][k] * B[k][j] for k in range(2)) for j in range(2)] for i in range(2)]

def matpow(M, n):
    """Puissance n-ieme par iterations (n >= 0)."""
    P = [[1, 0], [0, 1]]
    for _ in range(n):
        P = matmul(P, M)
    return P

I2 = [[1, 0], [0, 1]]
S = [[0, -1], [1, 0]]
T = [[1, 1], [0, 1]]
R = matmul(S, T)
Sinv = [[0, 1], [-1, 0]]           # S^{-1} = -S (car S^2 = -I)
moinsI = [[-1, 0], [0, -1]]

print("R = S*T          :", R == [[0, -1], [1, 1]])
print("T = S^{-1} * R   :", matmul(Sinv, R) == T)
print("S^4 = I          :", matpow(S, 4) == I2)
print("R^6 = I          :", matpow(R, 6) == I2)
print("S^2 = R^3 = -I   :", matpow(S, 2) == moinsI, matpow(R, 3) == moinsI)

R = S*T          : True
T = S^{-1} * R   : True
S^4 = I          : True
R^6 = I          : True
S^2 = R^3 = -I   : True True


### Lecture du résultat

- $S$ est d'ordre **4**, $R$ d'ordre **6** : $\langle S \rangle \cong C_4$ et $\langle R \rangle \cong C_6$.
- Le point clef est l'égalité mesurée $S^2 = R^3 = -I$ : les deux groupes cycliques **partagent**
  le même sous-groupe d'ordre 2, $\{\pm I\}$.
- C'est exactement la situation d'un **produit amalgamé** : ces mesures sont la signature
  combinatoire de l'isomorphisme de structure

$$SL_2(\mathbb{Z}) \;\cong\; C_4 *_{C_2} C_6,$$

l'amalgame se faisant au-dessus du $C_2$ central commun. La section 3 rendra cet arbre visible.

## 2. L'action sur $P^1(\mathbb{Q})$ et la descente de Stern-Brocot

La droite projective rationnelle $P^1(\mathbb{Q})$ est l'ensemble des droites de $\mathbb{Q}^2$ :
un point y est une classe de vecteur **primitif** $(p, q)$ (c'est-à-dire $\gcd(|p|, q) = 1$)
à signe près — $(p,q)$ et $(-p,-q)$ définissent le même point, et $(1,0)$ est le point à
l'infini. $SL_2(\mathbb{Z})$ agit par $M \cdot v = Mv$.

**Transitivité** : $SL_2(\mathbb{Z})$ envoie n'importe quel point sur n'importe quel autre.
Plutôt que de l'admettre, nous la **construisons** : la descente de Stern-Brocot localise tout
rationnel positif $p/q$ comme **médiante** d'un intervalle de Farey $(a/b, c/d)$ (avec
$|ad - bc| = 1$), atteint depuis l'intervalle racine $(0/1, \infty)$ par une suite de pas
gauche/droite — chaque pas « gauche » remplace l'extrémité droite par la médiante (matrice
$\tilde T = \left(\begin{smallmatrix} 1 & 0 \\ 1 & 1 \end{smallmatrix}\right)$), chaque pas
« droite » remplace l'extrémité gauche (matrice $T$). Les négatifs s'obtiennent en préfixant par
$S^2 = -I$, et $0$ vaut $S \cdot \infty$.

In [2]:
def norm(p, q):
    """Representant canonique du point projectif (p,q) ~ (-p,-q) :
    q > 0, ou q == 0 et p > 0 ; puis reduction par le pgcd."""
    if q < 0 or (q == 0 and p < 0):
        p, q = -p, -q
    g = gcd(p, q) or 1
    return (p // g, q // g)

Tt = [[1, 0], [1, 1]]   # T-tilde : pas gauche de Stern-Brocot

def point(M, v):
    """Image du vecteur v par M, normalisee en point projectif."""
    return norm(M[0][0] * v[0] + M[0][1] * v[1], M[1][0] * v[0] + M[1][1] * v[1])

def descente_sb(p, q):
    """Descente de Stern-Brocot vers p/q > 0.
    Renvoie (chemin, M) ou M envoie la racine (0/1, infini) sur l'intervalle
    (a/b, c/d) dont la mediante est p/q ; chemin = suite de 'G'/'D'."""
    cible = norm(p, q)
    u, v = (0, 1), (1, 0)
    M = I2
    chemin = ""
    for _ in range(400):
        m = (u[0] + v[0], u[1] + v[1])
        if m == cible:
            return chemin, M
        # p/q < mediante  ssi  p*m_q < m_p*q  (denominateurs positifs)
        if cible[0] * m[1] < m[0] * cible[1]:
            v, M, chemin = m, matmul(M, Tt), chemin + "G"
        else:
            u, M, chemin = m, matmul(M, T), chemin + "D"
    return None, None

# Verification exhaustive : tout rationnel de hauteur <= 40 est reconstruit exactement.
tous = sorted({norm(p, q) for p in range(-40, 41) for q in range(1, 41) if gcd(p, q) == 1})
ok, profondeurs = 0, []
for (p, q) in tous:
    if p == 0:
        continue                      # 0 = S.(infini), hors de la descente positive
    chemin, M = descente_sb(abs(p), q)
    med = point(matmul(M, T), (0, 1))  # la mediante est l'image de e2 par M*T
    atteint = med if p > 0 else norm(-med[0], med[1])
    if atteint == (p, q):
        ok += 1
        profondeurs.append(len(chemin))
print(f"Rationnels de hauteur <= 40 reconstruits exactement : {ok}/{len(tous) - 1} (0 exclu)")
print(f"Profondeur de descente : max {max(profondeurs)}, moyenne {sum(profondeurs)/len(profondeurs):.1f}")
print(f"Exemple 3/8  : chemin '{descente_sb(3, 8)[0]}'")
print(f"Exemple 5/7  : chemin '{descente_sb(5, 7)[0]}'")
print(f"Exemple 1/40 : chemin '{descente_sb(1, 40)[0]}' (le pire cas, fraction continue [0;40])")

Rationnels de hauteur <= 40 reconstruits exactement : 1958/1958 (0 exclu)
Profondeur de descente : max 39, moyenne 11.0
Exemple 3/8  : chemin 'GGDG'
Exemple 5/7  : chemin 'GDDG'
Exemple 1/40 : chemin 'GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG' (le pire cas, fraction continue [0;40])


### Lecture du résultat

**1958 rationnels sur 1958** (hauteur $\le 40$) sont reconstruits *exactement* — pour chacun,
le mot construit par la descente renvoie bien sa médiante sur la cible. La transitivité de
l'action n'est pas un argument invoqué, c'est un algorithme **exécuté** : profondeur moyenne
d'environ 11 pas, pire cas mesuré 39 (pour 1/40). La profondeur de descente de $p/q$ est la
**somme des coefficients de sa fraction continue, moins un** (pour 1/40 : $40 - 1 = 39$) :
chaque pas G ou D est un pas de l'algorithme d'Euclide, et l'arbre de Stern-Brocot est le livre
de comptes de cet algorithme.

In [3]:
def stern_brocot(n):
    """Toutes les mediantes de l'arbre de Stern-Brocot de hauteur <= n (LISTE, doublons visibles)."""
    vu = []
    pile = [((0, 1), (1, 0))]
    while pile:
        u, v = pile.pop()
        m = (u[0] + v[0], u[1] + v[1])
        if max(abs(m[0]), m[1]) > n:     # coupe par HAUTEUR (pas seulement denominateur !)
            continue
        vu.append(m)
        pile.append((u, m)); pile.append((m, v))
    return vu

mediantes = stern_brocot(40)
uniques = set(mediantes)
attendus = {(p, q) for q in range(1, 41) for p in range(1, q + 1) if gcd(p, q) == 1}
print(f"Mediantes de hauteur <= 40 : {len(mediantes)} noeuds, {len(uniques)} distincts")
print(f"Rationnels de ]0,1] couverts : {len(uniques & attendus)}/{len(attendus)}")
print(f"Doublons : {'oui' if len(mediantes) != len(uniques) else 'non'}")

Mediantes de hauteur <= 40 : 979 noeuds, 979 distincts
Rationnels de ]0,1] couverts : 490/490
Doublons : non


### Lecture du résultat

Chaque rationnel de $]0,1]$ apparaît **exactement une fois** comme médiante (490/490 couverts,
zéro doublon sur les 979 médiantes de hauteur $\le 40$) : l'arbre de Stern-Brocot énumère
$\mathbb{Q}_{>0}$ **sans répétition**. C'est la propriété qui fera de lui, à la section suivante,
le squelette d'un arbre au sens de Serre.

*Remarque d'implémentation* : la coupe de l'exploration doit se faire sur la **hauteur**
$max(|p|, q)$ — couper sur le seul dénominateur laisserait la branche $(k/1, \infty)$ pousser
sans fin, ses médiantes $(k+1)/1$ ayant toutes dénominateur 1.

## 3. L'arbre de Farey, arbre de Bass-Serre de l'amalgame

Le **pavage de Farey** est la triangulation idéale du demi-plan supérieur dont les sommets sont
les points de $P^1(\mathbb{Q})$ et dont les arêtes relient les paires **farey-adjacentes** :
$|ps - qr| = 1$. Chaque arête $\{u, v\}$ porte exactement deux triangles, de troisièmes sommets
$u + v$ et $u - v$ (les deux solutions de l'adjacence simultanée). Considérons le **graphe
d'incidence triangles ↔ arêtes** et mesurons ses bulles :

- si chaque arête porte exactement 2 triangles et chaque triangle 3 arêtes, le graphe est
  **birégulier (2, 3)** ;
- si dans toute bulle finie $|\text{triangles}| - |\text{arêtes internes}| = 1$, il est **connexe
  et acyclique** : un arbre.

C'est précisément l'arbre de Bass-Serre de l'écriture $C_4 *_{C_2} C_6$ : les sommets-triangles
(degré 3) portent les stabilisateurs d'ordre 6, les sommets-arêtes (degré 2) ceux d'ordre 4, et
$-I$ — central, fixant chaque sommet — est le $C_2$ sur lequel l'amalgame se fait.

In [4]:
from collections import deque

def bulle_farey(R):
    """BFS de profondeur R sur les triangles du pave de Farey.
    Racine : triangle (infini, 0/1, 1/1). L'arete {u,v} porte exactement 2 triangles :
    {u,v,u+v} et {u,v,u-v} (troisiemes sommets, en points projectifs)."""
    tri0 = frozenset({(1, 0), (0, 1), (1, 1)})
    tris = {tri0}
    front = deque([(tri0, 0)])
    while front:
        t, d = front.popleft()
        if d >= R:
            continue
        us = sorted(t)
        for i in range(3):
            for j in range(i + 1, 3):
                u, v = us[i], us[j]
                for w in ((u[0] + v[0], u[1] + v[1]), (u[0] - v[0], u[1] - v[1])):
                    if w == (0, 0):
                        continue
                    t2 = frozenset({u, v, norm(*w)})
                    if t2 not in tris:
                        tris.add(t2)
                        front.append((t2, d + 1))
    return tris

print(f"{'R':>2} {'triangles':>10} {'ar.internes':>11} {'frontiere':>10} "
      f"{'porteurs>=3':>12} {'|tri|-int':>9}")
for R in (4, 5, 6, 7):
    tris = bulle_farey(R)
    aretes = set()
    for t in tris:                      # toutes les aretes de la bulle FERMEE
        us = sorted(t)
        for i in range(3):
            for j in range(i + 1, 3):
                aretes.add(frozenset({us[i], us[j]}))
    internes, frontiere, trop = 0, 0, 0
    for e in aretes:
        n = sum(1 for t in tris if e <= t)
        if n == 2:
            internes += 1
        elif n == 1:
            frontiere += 1
        else:
            trop += 1
    print(f"{R:>2} {len(tris):>10} {internes:>11} {frontiere:>10} {trop:>12} "
          f"{len(tris) - internes:>9}")

 R  triangles ar.internes  frontiere  porteurs>=3 |tri|-int
 4         46          45         48            0         1
 5         94          93         96            0         1
 6        190         189        192            0         1
 7        382         381        384            0         1


### Lecture du résultat

Les trois propriétés d'un arbre sont **mesurées simultanément**, à chaque rayon :

1. **jamais 3 porteurs** — une arête porte exactement 2 triangles dans le pavage entier ;
   dans la bulle, une arête frontière n'en montre qu'un (son second est hors bulle) ;
2. **$|\text{triangles}| - |\text{arêtes internes}| = 1$** à tous les rayons — connexité +
   acyclicité : le graphe d'incidence est un **arbre** ;
3. les accroissements $48 \to 96 \to 192 \to 384$ de la frontière **doublent** à chaque
   rayon — la croissance exponentielle de l'arbre birégulier (2, 3), signature d'un groupe
   **virtuellement libre**.

C'est cet arbre que Serre met au centre du livre : agir sur un arbre, c'est couper l'espace en
combinatoire pure. La lecture groupe-théorique des degrés — 3 pour les triangles, 2 pour les
arêtes — relit $[C_6 : C_2] = 3$ et $[C_4 : C_2] = 2$ : les indices de l'amalgame sont les
degrés de l'arbre.

## 4. Le ping-pong de Sanov — la liberté jouée à deux ensembles

Venons-en au théorème. Posons

$$A = T^2 = \begin{pmatrix} 1 & 2 \\ 0 & 1 \end{pmatrix}, \qquad B = S\,T^{-2}\,S^{-1} = \begin{pmatrix} 1 & 0 \\ 2 & 1 \end{pmatrix}.$$

Ce sont les deux générateurs standard du **groupe principal de congruence** $\Gamma(2)$
(diagonales impaires, hors-diagonales paires). Le théorème de **Sanov (1947)** affirme que
$\langle A, B \rangle$ est **libre de rang 2**. La démonstration est un ping-pong sur la droite
projective réelle, avec les deux « camps »

$$X = \{t : |t| > 1\}, \qquad Y = \{t : |t| < 1\}.$$

Si $A^{\pm 1}$ envoie $Y$ dans $X$ et $B^{\pm 1}$ envoie $X$ dans $Y$, alors tout mot réduit non
vide en $A^{\pm 1}, B^{\pm 1}$ déplace une bille posée dans l'un des deux camps à travers les
camps alternés sans jamais revenir : il ne peut pas valoir l'identité. Mesurons les quatre
inclusions, point par point, en rationnels exacts :

In [5]:
def agit(M, z):
    """Action de M sur t dans la droite projective : t -> (a t + b)/(c t + d).
    z est un Fraction, None designe l'infini."""
    if z is None:
        return None if M[1][0] == 0 else Fraction(M[0][0], M[1][0])
    num = M[0][0] * z + M[0][1]
    den = M[1][0] * z + M[1][1]
    if den == 0:
        return None
    return num / den

A = matpow(T, 2)                                   # [[1,2],[0,1]]
B = matmul(matmul(S, [[1, -2], [0, 1]]), Sinv)      # S T^{-2} S^{-1} = [[1,0],[2,1]]
Ainv, Binv = [[1, -2], [0, 1]], [[1, 0], [-2, 1]]

def dans_gamma2(M):
    """Appartenance au groupe principal de congruence mod 2."""
    return M[0][0] % 2 == 1 and M[1][1] % 2 == 1 and M[0][1] % 2 == 0 and M[1][0] % 2 == 0

print("A ==", A, " B ==", B)
print("A, B dans Gamma(2) :", dans_gamma2(A), dans_gamma2(B))

# Grilles exhaustives de points rationnels de chaque camp (denominateur <= 12)
Y_pts = [Fraction(p, q) for q in range(1, 13) for p in range(-q + 1, q) if abs(Fraction(p, q)) < 1]
X_pts = [Fraction(p, q) for q in range(1, 13) for p in range(-30, 31) if abs(Fraction(p, q)) > 1]

okA = all(abs(agit(M, x)) > 1 for x in Y_pts for M in (A, Ainv))
okB = all(abs(agit(M, x)) < 1 for x in X_pts for M in (B, Binv))
print(f"A et A^(-1) envoient Y dans X : {okA}   ({len(Y_pts)} points)")
print(f"B et B^(-1) envoient X dans Y : {okB}   ({len(X_pts)} points)")
print("Sur la grille : min de |A(x)| sur Y  =", min(abs(agit(A, x)) for x in Y_pts))
print("Sur la grille : max de |B(x)| sur X  =", max(abs(agit(B, x)) for x in X_pts))

A == [[1, 2], [0, 1]]  B == [[1, 0], [2, 1]]
A, B dans Gamma(2) : True True
A et A^(-1) envoient Y dans X : True   (144 points)
B et B^(-1) envoient X dans Y : True   (564 points)
Sur la grille : min de |A(x)| sur Y  = 13/12
Sur la grille : max de |B(x)| sur X  = 13/14


### Lecture du résultat

Les quatre inclusions du ping-pong tiennent **sur chaque point testé**. Elles sont en fait des
inégalités rationnelles exactes : $A$ envoie $Y = (-1, 1)$ sur l'intervalle ouvert $(1, 3)$ et
$A^{-1}$ sur $(-3, -1)$ — tous deux dans $X$ ; $B$ agit par $t \mapsto t/(2t+1)$, et $|t| > 1$
entraîne $|2t+1| > |t|$, donc $|Bt| < 1$. La marge est mince aux frontières : sur la grille la
plus fine, $|A x|$ descend à $13/12$ (près de $t = -1$) et $|B x|$ monte à $13/14$ — les
inclusions sont **strictes** mais jamais larges, et le lemme n'exige que la stricte inclusion.
Par le ping-pong, **$\langle A, B \rangle$ est libre** : $\Gamma(2) \subset SL_2(\mathbb{Z})$
contient un groupe libre de rang 2, et $SL_2(\mathbb{Z})$ avec lui.

In [6]:
def mots_reduits(L):
    """Tous les mots reduits de longueur exactement L sur {A, B, a=A^-1, b=B^-1}
    (jamais une lettre suivie de son inverse)."""
    inv = {"A": "a", "a": "A", "B": "b", "b": "B"}
    res = []
    def rec(m):
        if len(m) == L:
            res.append(m)
            return
        for g in "ABab":
            if not m.endswith(inv[g]):
                rec(m + g)
    rec("")
    return res

GMAP = {"A": A, "B": B, "a": Ainv, "b": Binv}
total, non_identite = 0, 0
for L in range(1, 10):
    for mot in mots_reduits(L):
        M = I2
        for g in mot:
            M = matmul(M, GMAP[g])
        total += 1
        if M != I2:
            non_identite += 1
print(f"Mots reduits de longueur <= 9 en A, B : {total}, egaux a l'identite : {total - non_identite}")

commutateur = matmul(matmul(A, B), matmul(Ainv, Binv))
trace = commutateur[0][0] + commutateur[1][1]
print("[A, B] ==", commutateur, " trace :", trace)

Mots reduits de longueur <= 9 en A, B : 39364, egaux a l'identite : 0
[A, B] == [[21, -8], [8, -3]]  trace : 18


### Lecture du résultat

**39 364 mots réduits sur 39 364** (toutes les longueurs jusqu'à 9) sont distincts de
l'identité — la liberté n'est pas seulement prouvée par le ping-pong, elle est **constatée**
à cette profondeur. Le commutateur $[A, B]$ a une trace de 18 : $|\text{tr}| > 2$, il est
**hyperbolique** — ses puissances croissent comme une suite de Fibonacci de matrices : la
croissance exponentielle de l'arbre de la section 3, vue depuis le groupe.

## 5. Contrepoint — le couple $(T,\ S T S^{-1})$ n'est **pas** libre

La liberté n'est pas une propriété du groupe ambiant mais du couple choisi. Remplaçons $A, B$
par $L_1 = T$ et $L_2 = S T S^{-1}$ — deux matrices très ressemblantes (mêmes formes
$\left(\begin{smallmatrix} 1 & \pm 1 \\ 0 & 1 \end{smallmatrix}\right)$, tournées de quart
de tour). Le ping-pong échoue : $T$ envoie $Y = \{|t| < 1\}$ sur l'intervalle $(0, 2)$, qui
chevauche les deux camps. Que trouve l'énumération des mots réduits ?

In [7]:
L1 = T
L2 = matmul(matmul(S, T), Sinv)       # = [[1,0],[-1,1]]
L1inv, L2inv = [[1, -1], [0, 1]], [[1, 0], [1, 1]]

# 1. L'echec du ping-pong, mesure : T laisse dans Y une partie de Y (chevauchement des camps)
echec = [x for x in Y_pts if abs(agit(T, x)) <= 1]
print("Points de Y laisses dans Y par T :", len(echec), "- le ping-pong (X, Y) est impossible")

# 2. Enumeration : des mots reduits egaux a l'identite apparaissent
from collections import Counter
GMAP2 = {"A": L1, "B": L2, "a": L1inv, "b": L2inv}
total2, relations = 0, []
for L in range(1, 9):
    for mot in mots_reduits(L):
        M = I2
        for g in mot:
            M = matmul(M, GMAP2[g])
        total2 += 1
        if M == I2:
            relations.append(mot)
print(f"Mots reduits <= 8 pour (T, S T S^(-1)) : {total2}, relations trouvees : {len(relations)}")
print("Repartition par longueur :", dict(Counter(len(m) for m in relations)))
print("Premieres relations :", relations[:3])

# 3. La relation sous-jacente, verifiee d'un coup : la relation de tresse
print("T L T == L T L :", matmul(matmul(L1, L2), L1) == matmul(matmul(L2, L1), L2))

Points de Y laisses dans Y par T : 78 - le ping-pong (X, Y) est impossible


Mots reduits <= 8 pour (T, S T S^(-1)) : 13120, relations trouvees : 68
Repartition par longueur : {6: 12, 8: 56}
Premieres relations : ['ABAbab', 'ABabaB', 'AbabAB']
T L T == L T L : True


### Lecture du résultat

Le contraste est total : pour ce couple, des mots réduits valent l'identité, et la relation
sous-jacente se vérifie d'un coup :

$$T\,L\,T \;=\; L\,T\,L, \qquad L = S T S^{-1}.$$

C'est la **relation de tresse** à trois brins : il existe une surjection du groupe de tresses
$B_3$ (présenté par $\sigma_1\sigma_2\sigma_1 = \sigma_2\sigma_1\sigma_2$) sur $\langle T, L
\rangle$ — et le quotient de $B_3$ par son centre est précisément $PSL_2(\mathbb{Z})$. Même
groupe ambiant, deux couples : l'un est libre (Sanov), l'autre tresse. Le ping-pong est
exactement ce qui fait la différence — deux camps disjoints échangés interdisent les relations ;
des camps qui se chevauchent les laissent naître.

## 6. L'écho Lean

Ce carnet est du versant « calculé » de la série ; il n'ajoute aucune preuve au lake.

- **Lake de la série** (vérifié par examen direct du dépôt) : `serre100_lean/` ne contient aucun
  code dédié à $SL_2$ — la seule évocation est un commentaire du carnet-miroir `Serre100/Tour.lean`
  (la variance sous $SL_2(\mathbb{Z})$ du carnet 09). Rien à brancher ici.
- **Briques standard** : Mathlib porte les groupes spéciaux linéaires (`SpecialLinearGroup`,
  dont $SL(2, \cdot)$ générique) et les groupes libres (`FreeGroup`) — affirmer au-delà de ces
  noms exigerait de re-vérifier l'API exacte au moment d'une formalisation (les sources Mathlib
  ne sont pas matérialisées dans le clone local, seuls les oleans du cache de build y figurent).
- **Ce qui manque, à notre connaissance** : le lemme de ping-pong, le théorème de Sanov et la
  décomposition $SL_2(\mathbb{Z}) \cong C_4 *_{C_2} C_6$ n'ont pas d'équivalent Mathlib direct.
  La voie naturelle formaliserait l'action sur la droite projective $\mathbb{P}^1(\mathbb{Q})$,
  puis un ping-pong sur deux parties — grain futur pour le lake, pas du présent carnet.

En attendant, l'arbre de la section 3 reste calculé, pas démontré : c'est le partage de travail
assumé de la série.

## 7. Exemples guidés résolus et exercices

### Exemple guidé 1 — ordre d'un élément et classification par la trace

Dans $SL_2(\mathbb{Z})$, la trace classe presque tout : $|\text{tr}| < 2$ elliptique (ordre
fini), $|\text{tr}| > 2$ hyperbolique (ordre infini), $|\text{tr}| = 2$ parabolique (ordre
infini) — à l'exception du centre $\pm I$, où $-I$ (trace $-2$) est d'ordre fini 2. Les ordres
finis possibles sont exactement $1, 2, 3, 4, 6$.

In [8]:
from itertools import product

def ordre_fini(M, borne=13):
    """Ordre de M s'il est <= borne, sinon None (ordre infini ou tres grand)."""
    P = I2
    for n in range(1, borne + 1):
        P = matmul(P, M)
        if P == I2:
            return n
    return None

mots_courts = {}
for L in range(1, 5):
    for suite in product("ST", repeat=L):
        M = I2
        for g in suite:
            M = matmul(M, S if g == "S" else T)
        mots_courts["".join(suite)] = M

ordres_vus = set()
for M in mots_courts.values():
    if M == I2:
        continue                      # l'identite est d'ordre 1, sans interet ici
    n = ordre_fini(M)
    if n is not None:
        ordres_vus.add(n)
print("Ordres finis observes sur les elements non triviaux (mots <= 4 lettres en S, T) :",
      sorted(ordres_vus))
for mot in ("S", "T", "SS", "ST", "TS", "STST"):
    M = mots_courts[mot]
    tr = M[0][0] + M[1][1]
    n = ordre_fini(M)
    classe = ("elliptique" if n is not None
              else ("parabolique" if abs(tr) == 2 else "hyperbolique"))
    print(f"  {mot:>5} : trace {tr:>3}, ordre {n if n else 'infini'}, {classe}")

Ordres finis observes sur les elements non triviaux (mots <= 4 lettres en S, T) : [2, 3, 4, 6]
      S : trace   0, ordre 4, elliptique
      T : trace   2, ordre infini, parabolique
     SS : trace  -2, ordre 2, elliptique
     ST : trace   1, ordre 6, elliptique
     TS : trace   1, ordre 6, elliptique
   STST : trace  -1, ordre 3, elliptique


La classification se lit mot à mot : $S$ elliptique d'ordre 4, $S^2 = -I$ elliptique
d'ordre 2 (trace $-2$ : l'exception centrale), $R = ST$ elliptique d'ordre 6, $T$ parabolique
(trace 2, ordre infini), $STST = R^2$ elliptique d'ordre 3. Aucun ordre fini hors de
$\{2, 3, 4, 6\}$ n'apparaît parmi les éléments non triviaux issus de ces 30 mots — la liste
complète des ordres finis de $SL_2(\mathbb{Z})$, constatée sur cet échantillon.

### Exemple guidé 2 — la descente de 13/29, intervalle par intervalle

Suivons la descente de Stern-Brocot pas à pas : chaque ligne est un intervalle de Farey qui
**contient** encore la cible, et la médiante finale est la cible.

In [9]:
def fmt(pt):
    """Affichage d'un point projectif : infini ou p/q."""
    return "inf" if pt == (1, 0) else f"{pt[0]}/{pt[1]}"

cible_e2 = norm(13, 29)
u, v = (0, 1), (1, 0)
chemin_e2 = ""
print("Descente de 13/29 depuis l'intervalle racine (0/1, inf) :")
for _ in range(60):
    m = (u[0] + v[0], u[1] + v[1])
    if m == cible_e2:
        print(f"  ({fmt(u)}, {fmt(v)}) -> mediante {fmt(m)} = CIBLE")
        break
    pas = "G" if Fraction(cible_e2[0], cible_e2[1]) < Fraction(m[0], m[1]) else "D"
    print(f"  ({fmt(u)}, {fmt(v)}) -> mediante {fmt(m)} : pas {pas}")
    if pas == "G":
        v = m
    else:
        u = m
    chemin_e2 += pas
print("Chemin :", chemin_e2, " (longueur", len(chemin_e2), ")")
print("Coherence avec descente_sb :", descente_sb(13, 29)[0] == chemin_e2)

Descente de 13/29 depuis l'intervalle racine (0/1, inf) :
  (0/1, inf) -> mediante 1/1 : pas G
  (0/1, 1/1) -> mediante 1/2 : pas G
  (0/1, 1/2) -> mediante 1/3 : pas D
  (1/3, 1/2) -> mediante 2/5 : pas D
  (2/5, 1/2) -> mediante 3/7 : pas D
  (3/7, 1/2) -> mediante 4/9 : pas D
  (4/9, 1/2) -> mediante 5/11 : pas G
  (4/9, 5/11) -> mediante 9/20 : pas G
  (4/9, 9/20) -> mediante 13/29 = CIBLE
Chemin : GGDDDDGG  (longueur 8 )
Coherence avec descente_sb : True


La cible reste pincée dans chaque intervalle successif — les extrémités sont toujours
farey-adjacentes et la médiante finale **est** la cible : la preuve constructive de la
transitivité, jouée sur un exemple. Le chemin $GGDDDDGG$ a pour longueur 8 $=$ (somme des
coefficients de la fraction continue $[0; 2, 4, 3]$) $-$ 1.

### Exemple guidé 3 — marges du ping-pong pour $(A^2, B^2)$

Le couple des carrés $(A^2, B^2)$ est lui aussi libre. Mesurons ses marges : le ping-pong
devient **uniforme**.

In [10]:
A2, B2 = matpow(A, 2), matpow(B, 2)
A2inv, B2inv = [[1, -4], [0, 1]], [[1, 0], [-4, 1]]
okA2 = all(abs(agit(M, x)) > 1 for x in Y_pts for M in (A2, A2inv))
okB2 = all(abs(agit(M, x)) < 1 for x in X_pts for M in (B2, B2inv))
print(f"A^2 et A^-2 envoient Y dans X : {okA2}")
print(f"B^2 et B^-2 envoient X dans Y : {okB2}")
print("Sur la grille : min de |A^2(x)| sur Y =", min(abs(agit(A2, x)) for x in Y_pts))
print("Sur la grille : max de |B^2(x)| sur X =", max(abs(agit(B2, x)) for x in X_pts))

A^2 et A^-2 envoient Y dans X : True
B^2 et B^-2 envoient X dans Y : True
Sur la grille : min de |A^2(x)| sur Y = 37/12
Sur la grille : max de |B^2(x)| sur X = 13/40


Pour $(A, B)$, les marges dégénéraient à la frontière $t = -1$ ($|A t| \to 1$ et
$|B t| \to 1$). Pour les carrés, elles sont **uniformes** : $A^2$ envoie $Y = (-1,1)$ sur
$(3, 5)$ — à distance au moins 3 — et $|B^2 t| = |t / (4t + 1)| < 1/3$ pour tout $|t| > 1$
(l'inégalité $3|t| < |4t+1|$ se factorise en $(7t+1)(t+1) > 0$, vraie hors de $(-1, -1/7)$).
Sur la grille, le minimum de $|A^2 x|$ est $37/12$ et le maximum de $|B^2 x|$ est $13/40$.
Plus on prend de puissances, plus les camps sont séparés : $\langle A^2, B^2 \rangle$ est un
sous-groupe d'indice fini du groupe libre de Sanov — un groupe libre de rang 5, par le théorème
de Nielsen-Schreier.

### Exercice 1 — l'ordre d'un élément

Écrire une fonction `ordre_element(M)` qui renvoie l'ordre exact de `M` quand il est
$\leq 12$, et la chaîne `'infini'` sinon. Vérifier : `ordre_element(S) == 4`,
`ordre_element(R) == 6`, `ordre_element(T) == 'infini'`, et déterminer l'ordre de `T S`.

In [11]:
def ordre_element(M):
    """Renvoie l'ordre de M s'il est <= 12, la chaine 'infini' sinon.
    Indice : |trace| = 2 ou > 2 annonce un ordre infini (parabolique ou hyperbolique,
    sauf -I d'ordre 2) ; |trace| < 2 annonce un ordre fini parmi 2, 3, 4, 6."""
    # TODO etudiant
    return None  # a completer

# Verifications attendues :
# print(ordre_element(S), ordre_element(R), ordre_element(T), ordre_element(matmul(T, S)))
print("A completer : implementer ordre_element puis decommenter les verifications.")

A completer : implementer ordre_element puis decommenter les verifications.


### Exercice 2 — les voisins de Farey de $5/7$

Deux points $p/q$ et $r/s$ sont farey-adjacents quand $|ps - qr| = 1$. Écrire une fonction qui
liste **tous** les voisins $(r, s)$ de $5/7$ avec $|s| \leq 30$, vérifier l'adjacence de
chacun, et identifier lesquels apparaissent comme extrémités des intervalles de la descente de
$5/7$.

In [12]:
def voisins_farey(p, q, borne_s=30):
    """Liste des (r, s) avec |s| <= borne_s, (r,s) primitif, tels que |p*s - q*r| == 1."""
    # TODO etudiant
    return None  # a completer

# Verifications attendues :
# v = voisins_farey(5, 7)
# print(v)                     -- puis verifier chaque adjacence,
#                               et comparer aux intervalles visites par descente_sb(5, 7)
print("A completer : implementer voisins_farey puis decommenter les verifications.")

A completer : implementer voisins_farey puis decommenter les verifications.


### Exercice 3 — la liberté de $(A^2, B^2)$ par énumération

L'exemple guidé 3 prouve la liberté de $\langle A^2, B^2 \rangle$ par ping-pong ; vérifions-la
numériquement. Écrire une fonction qui énumère tous les mots réduits de longueur $\leq 7$ en
$A^2, B^2$ (et leurs inverses) et compte ceux qui valent l'identité — le compteur doit rester à
zéro.

In [13]:
def liberte_mesuree(M1, M2, M1inv, M2inv, Lmax=7):
    """Renvoie (nb_mots_equant_I, nb_mots_total) sur les mots reduits de longueur <= Lmax.
    S'appuie sur mots_reduits (section 4)."""
    # TODO etudiant
    return None  # a completer

# Verifications attendues :
# n_rel, n_tot = liberte_mesuree(A2, B2, [[1, -4], [0, 1]], [[1, 0], [-4, 1]])
# print(f"{n_rel} relation(s) sur {n_tot} mots reduits")
print("A completer : implementer liberte_mesuree puis decommenter les verifications.")

A completer : implementer liberte_mesuree puis decommenter les verifications.


## Conclusion

Ce carnet a fait mesurer, par des sorties exactes, les trois étages d'« Arbres, amalgames et
SL₂ » :

1. **La signature** $S^4 = R^6 = I$, $S^2 = R^3 = -I$ — deux cycliques partageant un $C_2$ :
   l'amalgame $C_4 *_{C_2} C_6$ ;
2. **L'arbre** — le pavage de Farey, relu comme graphe d'incidence triangles/arêtes :
   birégulier (2, 3), acyclique (mesuré à quatre rayons), à frontière doublant à chaque rayon ;
   la descente de Stern-Brocot y construit la transitivité de l'action sur $P^1(\mathbb{Q})$
   (1958 rationnels reconstruits exactement) ;
3. **La liberté** — le ping-pong de Sanov, ses quatre inclusions mesurées point à point, la
   liberté constatée sur 39 364 mots réduits, et le contrepoint du couple $(T, S T S^{-1})$ où
   la même énumération découvre la relation de tresse $TLT = LTL$ au lieu de la liberté.

$SL_2(\mathbb{Z})$ n'est pas libre — $S^4 = I$ l'interdit — mais il **agit sur un arbre** avec
des stabilisateurs finis : il est virtuellement libre, et $\Gamma(2)$ en est la partie libre de
rang 2. Le ping-pong est le pont exact entre les deux affirmations : un groupe qui échange deux
camps disjoints contient une forêt de mots sans aucune relation.